# 02 — EDA & the 7 dashboard metrics

**Why (lessons 1.8–1.10):** compute every metric the Streamlit dashboard shows,
but visualise with **Matplotlib + Seaborn** here. Streamlit uses **Plotly**.
Seeing both is the lesson.

In [ ]:
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')
con = duckdb.connect('../data/sg_jobs.duckdb', read_only=True)
jobs = con.execute('''
  SELECT c.*, jc.category_name
  FROM clean_jobs c
  LEFT JOIN job_categories jc ON c.metadata_jobPostId = jc.job_post_id
''').df()
jobs.shape

## Metric 1 — applications per category
**Why:** rank categories by total applications to spot in-demand roles.

In [ ]:
m1 = (jobs.groupby('category_name')['metadata_totalNumberJobApplication']
          .sum().sort_values(ascending=False).head(12))
plt.figure(figsize=(9,5)); sns.barplot(x=m1.values, y=m1.index, palette='Blues_r')
plt.title('Top categories by total applications'); plt.tight_layout(); plt.show()

## Metrics 2 & 3 — vacancies + mean salary; under-hired categories
**Why:** vacancies high + applications low = shortage (mismatch < 1).

In [ ]:
g = jobs.groupby('category_name').agg(
    vacancies=('numberOfVacancies','sum'),
    apps=('metadata_totalNumberJobApplication','sum'),
    mean_salary=('average_salary','mean'))
g['mismatch'] = g['apps'] / g['vacancies'].replace(0, np.nan)
g.sort_values('mismatch').head(12)

## Metrics 4 & 5 — open days & reposts
**Why:** long open-days + high repost counts = chronically hard-to-fill roles.

In [ ]:
g2 = jobs.groupby('category_name').agg(
    open_days=('open_days','mean'),
    reposts=('metadata_repostCount','mean')).dropna()
g2.sort_values('open_days', ascending=False).head(10)

## Metric 6 — wage percentiles (PMET) vs EP thresholds
**Why (lesson 1.6 NumPy percentiles):** EP is benchmarked to the top third of PMET wages.
Plot p33/p50/p65/p90 per category to sanity-check the $5,600 / $6,200 EP bar.

In [ ]:
pmet = jobs[jobs['is_pmet']=='PMET']
pct = (pmet.groupby('category_name')['average_salary']
       .agg(p33=lambda s: np.percentile(s.dropna(),33),
            p50='median',
            p65=lambda s: np.percentile(s.dropna(),65),
            p90=lambda s: np.percentile(s.dropna(),90)))
pct.sort_values('p50', ascending=False).head(12)

## Metric 7 — Talent Scarcity & Mismatch Index
**Why:** apps/vacancies < 1 signals shortage → informs pass-quota pressure.

In [ ]:
jobs_pt = jobs[~jobs['is_part_time']]
idx = (jobs_pt.groupby('category_name')
       .apply(lambda d: d['metadata_totalNumberJobApplication'].sum()
                       / max(d['numberOfVacancies'].sum(),1))
       .sort_values())
idx.head(12)